In [37]:
import copy
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score
)
import joblib
import json
import torch.onnx

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [38]:
def load_and_preprocess(gen_path, weather_path):
    df_gen = pd.read_csv(gen_path)
    df_weather = pd.read_csv(weather_path)

    df_gen['DATE_TIME'] = pd.to_datetime(df_gen['DATE_TIME'], format='%d-%m-%Y %H:%M')
    df_weather['DATE_TIME'] = pd.to_datetime(df_weather['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')

    df_gen_agg = df_gen.groupby('DATE_TIME')[['DC_POWER', 'AC_POWER']].sum().reset_index()
    df_weather_agg = df_weather.groupby('DATE_TIME')[['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']].mean().reset_index()

    df_merged = pd.merge(df_gen_agg, df_weather_agg, on='DATE_TIME', how='inner')
    df_merged = df_merged.sort_values('DATE_TIME').reset_index(drop=True)

    df_merged['hour'] = df_merged['DATE_TIME'].dt.hour + df_merged['DATE_TIME'].dt.minute / 60.0
    df_merged['sin_hour'] = np.sin(2 * np.pi * df_merged['hour'] / 24.0)
    df_merged['cos_hour'] = np.cos(2 * np.pi * df_merged['hour'] / 24.0)
    df_merged = df_merged.drop(columns=['hour'])

    return df_merged

def chronological_split(df, lookback, horizon, train_frac=0.70, val_frac=0.15):
    n = len(df)
    gap = lookback + horizon

    train_end = int(n * train_frac)
    val_end = train_end + int(n * val_frac)

    df_train = df.iloc[:train_end].reset_index(drop=True)
    df_val = df.iloc[train_end + gap: val_end].reset_index(drop=True)
    df_test = df.iloc[val_end + gap:].reset_index(drop=True)

    return df_train, df_val, df_test

In [39]:
class MultiTaskSolarDataset(Dataset):
    def __init__(self, data, feature_cols, target_col, lookback_window, forecast_horizon, active_threshold=0.1, feature_scaler=None, target_scaler=None):
        self.lookback_window = lookback_window
        self.forecast_horizon = forecast_horizon
        
        raw_target_values = data[[target_col]].values
        self.target_binary = torch.tensor((raw_target_values > active_threshold).astype(float), dtype=torch.float32)

        if feature_scaler is None:
            self.feature_scaler = StandardScaler()
            self.features = self.feature_scaler.fit_transform(data[feature_cols].values)
        else:
            self.feature_scaler = feature_scaler
            self.features = self.feature_scaler.transform(data[feature_cols].values)

        if target_scaler is None:
            self.target_scaler = StandardScaler()
            self.target = self.target_scaler.fit_transform(raw_target_values)
        else:
            self.target_scaler = target_scaler
            self.target = self.target_scaler.transform(raw_target_values)

        self.features = torch.tensor(self.features, dtype=torch.float32)
        self.target_scaled = torch.tensor(self.target, dtype=torch.float32)
        self.length = max(0, len(data) - lookback_window - forecast_horizon + 1)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        start_in = idx
        end_in = start_in + self.lookback_window
        x = self.features[start_in:end_in]

        start_out = end_in
        end_out = start_out + self.forecast_horizon
        
        y_scaled = self.target_scaled[start_out:end_out]
        y_binary = self.target_binary[start_out:end_out]

        return x, y_scaled, y_binary

In [40]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MultiTaskSolarForecaster(nn.Module):
    def __init__(self, num_features, d_model, nhead, num_layers, dim_feedforward, forecast_horizon, lookback_window, dropout=0.22):
        super().__init__()
        self.forecast_horizon = forecast_horizon

        self.feature_embedding = nn.Linear(num_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.reg_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(d_model * lookback_window, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, forecast_horizon)
        )
        
        self.cls_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(d_model * lookback_window, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, forecast_horizon),
            nn.Sigmoid()
        )

    def forward(self, src):
        x = self.feature_embedding(src)
        x = self.pos_encoder(x)
        latent_features = self.transformer_encoder(x)
        
        reg_output = self.reg_head(latent_features)
        cls_output = self.cls_head(latent_features)
        
        return reg_output.unsqueeze(-1), cls_output.unsqueeze(-1)

In [41]:
def run_epoch(model, dataloader, criterion_reg, criterion_cls, optimizer=None, clip_norm=1.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    context = torch.enable_grad() if is_train else torch.no_grad()
    
    with context:
        for x_batch, y_reg, y_cls in dataloader:
            x_batch, y_reg, y_cls = x_batch.to(device), y_reg.to(device), y_cls.to(device)

            if is_train:
                optimizer.zero_grad()

            preds_reg, preds_cls = model(x_batch)
            
            loss_reg = criterion_reg(preds_reg, y_reg)
            loss_cls = criterion_cls(preds_cls, y_cls)
            loss = loss_reg + 0.8 * loss_cls

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)
                optimizer.step()

            total_loss += loss.item()

    return total_loss / len(dataloader)


def train_model(model, train_loader, val_loader, optimizer, epochs=50, patience=8, warmup_epochs=3):
    criterion_reg = nn.HuberLoss(delta=1.0)
    criterion_cls = nn.BCELoss()
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
    )


    best_val_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0

    for epoch in range(epochs):
        train_loss = run_epoch(model, train_loader, criterion_reg, criterion_cls, optimizer)
        val_loss = run_epoch(model, val_loader, criterion_reg, criterion_cls, optimizer=None)


        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.2e}")

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}.")
                break

    model.load_state_dict(best_state)
    print(f"Restored best weights (Val Loss: {best_val_loss:.4f}).")
    return model

def evaluate_and_print_metrics(model, dataloader, device, target_scaler, split_name="TEST"):
    model.eval()
    all_preds_reg, all_targets_reg = [], []
    all_preds_cls, all_targets_cls = [], []

    with torch.no_grad():
        for x_batch, y_reg, y_cls in dataloader:
            x_batch = x_batch.to(device)

            preds_reg, preds_cls = model(x_batch)
            all_preds_reg.append(preds_reg.squeeze(-1).cpu().numpy())
            all_targets_reg.append(y_reg.squeeze(-1).cpu().numpy())
            
            all_preds_cls.append(preds_cls.squeeze(-1).cpu().numpy())
            all_targets_cls.append(y_cls.squeeze(-1).cpu().numpy())

    all_preds_reg = np.concatenate(all_preds_reg).reshape(-1, 1)
    all_targets_reg = np.concatenate(all_targets_reg).reshape(-1, 1)

    all_preds_inv = target_scaler.inverse_transform(all_preds_reg).flatten()
    all_targets_inv = target_scaler.inverse_transform(all_targets_reg).flatten()
    all_preds_inv = np.maximum(all_preds_inv, 0)

    mae = mean_absolute_error(all_targets_inv, all_preds_inv)
    rmse = np.sqrt(mean_squared_error(all_targets_inv, all_preds_inv))
    r2 = r2_score(all_targets_inv, all_preds_inv)

    binary_targets = np.concatenate(all_targets_cls).flatten()
    binary_preds = (np.concatenate(all_preds_cls).flatten() > 0.5).astype(int)

    accuracy = accuracy_score(binary_targets, binary_preds)
    precision = precision_score(binary_targets, binary_preds, average='macro', zero_division=0)
    recall = recall_score(binary_targets, binary_preds, average='macro', zero_division=0)
    f1 = f1_score(binary_targets, binary_preds, average='macro', zero_division=0)

    target_mean = all_targets_inv.mean()
    
    print("="*55)
    print(f"   MULTI-TASK SOLAR PERFORMANCE REPORT — {split_name}")
    print("="*55)
    print(" REGRESSION METRICS (Continuous Power Output)")
    print("-" * 55)
    print(f" Mean Absolute Error (MAE):      {mae:.4f} kW ({100*mae/target_mean:.1f}% of mean)")
    print(f" Root Mean Squared Error (RMSE): {rmse:.4f} kW")
    print(f" R² Score:                       {r2:.4f}")
    print("\n CLASSIFICATION METRICS (Operational State Prediction)")
    print("-" * 55)
    print(f" Accuracy:  {accuracy * 100:.2f}%")
    print(f" Precision: {precision:.4f}")
    print(f" Recall:    {recall:.4f}")
    print(f" F1-Score:  {f1:.4f}")
    print("="*55)

    return {"mae": mae, "rmse": rmse, "r2": r2, "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

In [42]:
FEATURE_COLS = ['DC_POWER', 'AC_POWER', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'sin_hour', 'cos_hour']
TARGET_COL = 'AC_POWER'
LOOKBACK = 96
HORIZON = 16
BATCH_SIZE = 32

df_clean = load_and_preprocess('Plant_1_Generation_Data.csv', 'Plant_1_Weather_Sensor_Data.csv')

df_train, df_val, df_test = chronological_split(df_clean, LOOKBACK, HORIZON, train_frac=0.70, val_frac=0.15)

train_dataset = MultiTaskSolarDataset(df_train, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON, active_threshold=0.1)
val_dataset = MultiTaskSolarDataset(
    df_val, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON, active_threshold=0.1,
    feature_scaler=train_dataset.feature_scaler, target_scaler=train_dataset.target_scaler
)
test_dataset = MultiTaskSolarDataset(
    df_test, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON, active_threshold=0.1,
    feature_scaler=train_dataset.feature_scaler, target_scaler=train_dataset.target_scaler
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = MultiTaskSolarForecaster(
    num_features=len(FEATURE_COLS),
    d_model=64,
    nhead=4,
    num_layers=3,
    dim_feedforward=256,
    forecast_horizon=HORIZON,
    lookback_window=LOOKBACK,
    dropout=0.2
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

model = train_model(model, train_loader, val_loader, optimizer, epochs=50, patience=6)  # no warmup

test_metrics = evaluate_and_print_metrics(model, test_loader, device, train_dataset.target_scaler, split_name="TEST (held-out)")

torch.save(model.state_dict(), "solar_transformer_forecaster_multitask.pth")
joblib.dump(train_dataset.feature_scaler, "solar_transformer_feature_scaler.pkl")
joblib.dump(train_dataset.target_scaler, "solar_transformer_target_scaler.pkl")

config = {
    "num_features": len(FEATURE_COLS),
    "d_model": 64,
    "nhead": 4,
    "num_layers": 3,
    "dim_feedforward": 256,
    "forecast_horizon": HORIZON,
    "lookback_window": LOOKBACK,
    "dropout": 0.22,
    "weight_decay": 2.5e-4,
    "warmup_epochs": 3,
    "lr_scheduler_patience": 5,
    "early_stop_patience": 8,
    "loss_weight_cls": 0.8
}
with open("solar_model_multitask_config.json", "w") as f:
    json.dump(config, f)

model.eval()
dummy_input = torch.randn(1, LOOKBACK, len(FEATURE_COLS)).to(device)
onnx_path = "solar_transformer_multitask.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input_window'],
    output_names=['power_forecast', 'state_probability'],
    dynamic_axes={
        'input_window': {0: 'batch_size'},
        'power_forecast': {0: 'batch_size'},
        'state_probability': {0: 'batch_size'}
    }
)

Epoch 1/50 | Train Loss: 0.3857 | Val Loss: 0.1122 | LR: 1.00e-03
Epoch 2/50 | Train Loss: 0.1987 | Val Loss: 0.1229 | LR: 1.00e-03
Epoch 3/50 | Train Loss: 0.1740 | Val Loss: 0.0802 | LR: 1.00e-03
Epoch 4/50 | Train Loss: 0.1623 | Val Loss: 0.0909 | LR: 1.00e-03
Epoch 5/50 | Train Loss: 0.1380 | Val Loss: 0.0732 | LR: 1.00e-03
Epoch 6/50 | Train Loss: 0.1237 | Val Loss: 0.0951 | LR: 1.00e-03
Epoch 7/50 | Train Loss: 0.1185 | Val Loss: 0.1251 | LR: 1.00e-03
Epoch 8/50 | Train Loss: 0.1092 | Val Loss: 0.1037 | LR: 1.00e-03
Epoch 9/50 | Train Loss: 0.1020 | Val Loss: 0.1038 | LR: 1.00e-03
Epoch 10/50 | Train Loss: 0.1035 | Val Loss: 0.0751 | LR: 1.00e-03
Epoch 11/50 | Train Loss: 0.0908 | Val Loss: 0.1377 | LR: 1.00e-03
Early stopping at epoch 11.
Restored best weights (Val Loss: 0.0732).
   MULTI-TASK SOLAR PERFORMANCE REPORT — TEST (held-out)
 REGRESSION METRICS (Continuous Power Output)
-------------------------------------------------------
 Mean Absolute Error (MAE):      1809.7245 